# 03. PD Logistic Regression

The target models probability of good standing (`good_bad = 1`). Default probability is calculated in the next notebook as `1 - P(good)`.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

DATA_PATH = Path('../data/loan_data_2007_2014.csv')
data = pd.read_csv(DATA_PATH, low_memory=False)
bad_statuses = {'Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off', 'Late (31-120 days)'}
data['good_bad'] = (~data['loan_status'].isin(bad_statuses)).astype(int)

# Pre-specified handoff from notebook 02; WoE/IV did not select these variables.
refined_raw_features = ['grade', 'term', 'verification_status', 'int_rate', 'dti']
broad_raw_features = refined_raw_features + [
    'annual_inc', 'emp_length', 'home_ownership', 'purpose', 'addr_state', 'initial_list_status'
]
y = data['good_bad'].copy()
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    data[broad_raw_features].copy(), y, test_size=0.2, random_state=42, stratify=y
)

def prepare_design(train_raw, test_raw, numeric_features, categorical_features, reference_levels):
    train = train_raw.copy()
    test = test_raw.copy()
    training_medians = train[numeric_features].median()
    train[numeric_features] = train[numeric_features].fillna(training_medians)
    test[numeric_features] = test[numeric_features].fillna(training_medians)
    training_means = train[numeric_features].mean()
    training_stds = train[numeric_features].std().replace(0, 1)
    train[numeric_features] = (train[numeric_features] - training_means) / training_stds
    test[numeric_features] = (test[numeric_features] - training_means) / training_stds
    for column in categorical_features:
        train[column] = train[column].fillna('Missing').astype(str)
        test[column] = test[column].fillna('Missing').astype(str)
        if reference_levels[column] not in set(train[column]):
            raise ValueError(f"Reference {reference_levels[column]!r} is absent from training {column}")
    train = pd.get_dummies(train, columns=categorical_features, dtype=float)
    test = pd.get_dummies(test, columns=categorical_features, dtype=float).reindex(columns=train.columns, fill_value=0.0)
    reference_columns = [f'{column}_{level}' for column, level in reference_levels.items()]
    missing_references = sorted(set(reference_columns) - set(train.columns))
    if missing_references:
        raise ValueError(f'Missing reference columns: {missing_references}')
    return train.drop(columns=reference_columns), test.drop(columns=reference_columns), training_medians, training_means, training_stds

refined_reference_levels = {
    'grade': 'G', 'term': ' 60 months', 'verification_status': 'Not Verified',
}
broad_reference_levels = {
    **refined_reference_levels, 'emp_length': 'Missing', 'home_ownership': 'OTHER',
    'purpose': 'other', 'addr_state': 'AK', 'initial_list_status': 'f',
}
refined_X_train, refined_X_test, training_medians, training_means, training_stds = prepare_design(
    X_train_raw[refined_raw_features], X_test_raw[refined_raw_features],
    ['int_rate', 'dti'], ['grade', 'term', 'verification_status'], refined_reference_levels,
)
broad_X_train, broad_X_test, _, _, _ = prepare_design(
    X_train_raw, X_test_raw, ['int_rate', 'dti', 'annual_inc'],
    ['grade', 'term', 'verification_status', 'emp_length', 'home_ownership', 'purpose', 'addr_state', 'initial_list_status'],
    broad_reference_levels,
)

## Broader comparison and refined inference

The broader sklearn model is shown only as a held-out discrimination comparison. The refined model is an unpenalized, full-rank statsmodels Logit model: every categorical variable has an explicit reference level (`grade_G`, `term_ 60 months`, and `verification_status_Not Verified`). Numeric coefficients are per one training-standard-deviation increase, because their means and standard deviations are estimated from training rows only.

In [2]:
# The sklearn models are discrimination-only; no inferential statistics are derived from them.
broad_model = LogisticRegression(max_iter=1000, solver='liblinear').fit(broad_X_train, y_train)
broad_auc = roc_auc_score(y_test, broad_model.predict_proba(broad_X_test)[:, 1])

refined_sklearn = LogisticRegression(max_iter=1000, solver='liblinear').fit(refined_X_train, y_train)
refined_sklearn_auc = roc_auc_score(y_test, refined_sklearn.predict_proba(refined_X_test)[:, 1])

refined_design_train = sm.add_constant(refined_X_train, has_constant='add')
refined_design_test = sm.add_constant(refined_X_test, has_constant='add')
refined_result = sm.Logit(y_train, refined_design_train).fit(disp=False)
refined_prob_good = refined_result.predict(refined_design_test)
refined_auc = roc_auc_score(y_test, refined_prob_good)
pd.DataFrame({
    'model': ['broader sklearn (discrimination only)', 'refined sklearn (discrimination only)', 'refined statsmodels Logit'],
    'AUC': [broad_auc, refined_sklearn_auc, refined_auc],
    'Gini': [2 * broad_auc - 1, 2 * refined_sklearn_auc - 1, 2 * refined_auc - 1],
})

,model,AUC,Gini
0,broader sklearn (discrimination only),0.676154,0.352308
1,refined sklearn (discrimination only),0.657623,0.315246
2,refined statsmodels Logit,0.657617,0.315234


In [3]:
coefficients = pd.DataFrame({
    'feature': refined_result.params.index,
    'coefficient': refined_result.params.values,
    'standard_error': refined_result.bse.values,
    'p_value': refined_result.pvalues.values,
})
coefficients['odds_ratio'] = np.exp(coefficients['coefficient'])
coefficients['unit'] = np.where(
    coefficients['feature'].isin(['int_rate', 'dti']), 'per one training-standard-deviation increase',
    np.where(coefficients['feature'].eq('const'), 'reference-category intercept', 'versus explicit reference level'),
)
coefficients.sort_values('p_value')

,feature,coefficient,standard_error,p_value,odds_ratio,unit
0,const,2.255317,0.063973,2.943600e-272,9.538313,reference-category intercept
1,int_rate,-0.499219,0.017884,1.809868e-171,0.607005,per one training-standard-deviation increase
2,dti,-0.071402,0.005331,6.574170e-41,0.931088,per one training-standard-deviation increase
9,term_ 36 months,-0.103298,0.012936,1.400136e-15,0.901859,versus explicit reference level
10,verification_status_Source Verified,0.072871,0.014497,4.994689e-07,1.075592,versus explicit reference level
3,grade_A,0.314532,0.086353,2.700951e-04,1.369618,versus explicit reference level
11,verification_status_Verified,-0.041081,0.013904,3.130334e-03,0.959752,versus explicit reference level
6,grade_D,-0.056261,0.054404,3.010703e-01,0.945292,versus explicit reference level
8,grade_F,0.049634,0.049661,3.175742e-01,1.050886,versus explicit reference level
5,grade_C,-0.036266,0.061668,5.564736e-01,0.964384,versus explicit reference level


In [4]:
# The unpenalized full-rank Logit fit supplies these Wald p-values; no penalized sklearn pseudoinverse p-values are presented.
assert refined_design_train.shape[1] == np.linalg.matrix_rank(refined_design_train.to_numpy())
print('Full-rank refined design verified.')
print('Reference levels:', refined_reference_levels)
print(f'Refined held-out AUC: {refined_auc:.6f}; Gini: {2 * refined_auc - 1:.6f}')

Full-rank refined design verified.
Reference levels: {'grade': 'G', 'term': ' 60 months', 'verification_status': 'Not Verified'}
Refined held-out AUC: 0.657617; Gini: 0.315234


## Next stage

Notebook 04 uses this same train-only preprocessing, reference levels, and unpenalized refined statsmodels specification. It converts `P(good)` to PD, reports held-out AUC/Gini and a confusion matrix, then maps the training-anchored refined logit to an illustrative 300–850 scorecard.